  Vessel Flag Change Frequency Analysis (Pre- vs. Post-2023)

In [1]:
import pandas as pd
from requests import head

summary_df = pd.read_csv('../data/vessel_summary.csv')
identities_df = pd.read_csv('../data/vessel_identities.csv')

In [2]:
ais_df = identities_df[identities_df['source'] == 'self_reported'].copy()

ais_df['transmission_from'] = pd.to_datetime(
    ais_df['transmission_from'], format='mixed', utc=True
)
ais_df['transmission_to'] = pd.to_datetime(
    ais_df['transmission_to'], format='mixed', utc=True
)

In [3]:
# Standardize ship names (clean non-breaking spaces and irregular whitespace)
identities_df['shipname'] = (
    identities_df['shipname']
    .astype(str)
    .str.replace('\xa0', ' ')
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

Most popular flags:

In [4]:
identities_df['flag'].value_counts()

flag
PAN    33
MHL    32
RUS    32
LBR    25
MLT    13
GRC    12
CMR    11
SLE     7
HKG     6
SGP     6
BHS     5
BRB     5
MDG     4
GIN     4
PLW     4
AZE     4
CHN     4
TUR     4
COM     3
KWT     2
GBR     2
NIC     2
COK     2
ITA     2
GNQ     2
CYP     2
CAN     2
DNK     2
CYM     2
DEU     2
SAU     2
IMN     1
FRA     1
BES     1
MOZ     1
VUT     1
SMR     1
ESP     1
ATG     1
SYR     1
Name: count, dtype: int64

In [5]:
#Ships(imo) who changed the flag the most
flag_changes = identities_df.groupby('imo')['flag'].nunique().reset_index()
flag_changes.columns = ['imo', 'num_flags']
flag_changes = flag_changes.merge(summary_df[['imo', 'current_name']], on='imo')
flag_changes = flag_changes.sort_values('num_flags', ascending=False)
flag_changes.head(10)

,imo,num_flags,current_name
8,9282481,9,TAGOR
3,9213296,8,TELESTO
6,9236248,8,FENGHUANG
9,9290373,7,RAVELIN
5,9233777,6,OCEAN II
17,9328170,6,AETHER
11,9296377,6,MARJORIE
10,9292199,6,929-21%-99%
29,9412452,6,TSOIZIT
4,9224453,5,TRACOS


 Intelligence:look at the flag-change patterns for the top 5 most "suspicious" vessels before calculating the metric for the entire fleet.

In [6]:
top_imo = flag_changes.head(5)['imo'].tolist()
cols = ['imo', 'shipname', 'flag', 'transmission_from', 'transmission_to']
timeline = (
    identities_df[identities_df['imo'].isin(top_imo)][cols]
    .sort_values(['imo', 'transmission_from'])
)
timeline['transmission_from'] = pd.to_datetime(timeline['transmission_from'], format='mixed')
timeline['days_since_last_update'] = timeline.groupby('imo')['transmission_from'].diff()
timeline

,imo,shipname,flag,transmission_from,transmission_to,days_since_last_update
97,9213296,OVERSEAS SHIRLEY,CAN,2012-01-02 16:53:24+00:00,2012-10-09T14:54:10Z,NaT
96,9213296,OVERSEAS SHIRLEY,MHL,2012-10-23 05:49:54+00:00,2019-04-13T06:06:08Z,294 days 12:56:30
95,9213296,SEAWAYSSHIRLEY,MHL,2017-10-02 21:34:28+00:00,2018-11-01T14:10:29Z,1805 days 15:44:34
100,9213296,US SHIRLEY,COM,2018-11-02 02:54:00+00:00,2018-12-01T07:41:09Z,395 days 05:19:32
98,9213296,SHIRLEY,LBR,2018-12-01 07:58:07+00:00,2019-06-12T23:58:13Z,29 days 05:04:07
103,9213296,BRIGHT SONIA,PAN,2019-05-18 08:18:21+00:00,2024-10-02T12:45:54Z,168 days 00:20:14
102,9213296,BRIGHTSONIA,PAN,2019-05-18 08:39:13+00:00,2024-10-02T11:38:42Z,0 days 00:20:52
94,9213296,TELESTO,GIN,2024-10-02 12:46:42+00:00,2025-01-09T03:24:35Z,1964 days 04:07:29
99,9213296,TELESTO,BRB,2025-01-09 03:28:03+00:00,2025-11-22T10:48:56Z,98 days 14:41:21
101,9213296,TELESTO,PLW,2025-11-22 20:29:06+00:00,2026-09-08T19:47:29Z,317 days 17:01:03


In [7]:
#transform char to datatime
identities_df['transmission_from'] = pd.to_datetime(identities_df['transmission_from'], format='mixed')
identities_df = identities_df.sort_values(['imo', 'transmission_from'])
identities_df

,imo,source,shipname,flag,ssvid,match_fields,transmission_from,transmission_to,is_verified
170,9037123,self_reported,HAI CHANG DA LIAN,CHN,413202420,SEVERAL_FIELDS,2012-01-03 00:56:11+00:00,2015-05-26T10:00:35Z,True
169,9037123,self_reported,SHAN GANG RONG HE,CHN,413366530,SEVERAL_FIELDS,2015-05-26 10:21:35+00:00,2024-07-24T02:20:46Z,True
168,9037123,registry,ZHONGGANGDALIAN,CHN,413366530,REGISTRY,2015-05-26 10:38:39+00:00,2019-12-14T08:37:28Z,True
173,9037123,self_reported,LIAOYINGYU 35669,NaN,415555555,NO_MATCH,2018-12-14 18:03:42+00:00,2022-09-21T08:08:42.25Z,False
172,9037123,self_reported,ZHONG GANG DA LIAN,MHL,538090558,NO_MATCH,2019-12-14 09:12:33+00:00,2021-09-29T22:51:17Z,False
...,...,...,...,...,...,...,...,...,...
234,9867621,self_reported,ARISTOFANIS,MHL,538008714,NO_MATCH,2019-12-10 07:14:52+00:00,2024-09-17T03:01:32Z,False
235,9867621,registry,KHANKENDI,AZE,423543100,REGISTRY,2024-08-23 07:44:30+00:00,2026-07-31T13:30:53Z,True
236,9867621,self_reported,KHANKENDI,AZE,423543100,SEVERAL_FIELDS,2024-08-23 07:44:37+00:00,2026-09-08T23:19:51Z,True
243,9884150,self_reported,SERGEY LVOV,RUS,273295620,NO_MATCH,2021-08-31 10:22:36+00:00,2026-09-08T23:26:26Z,False


In [8]:
#Added column to understand time elapsed since last shipname update
identities_df['days_since_last_update'] = identities_df.groupby('imo')['transmission_from'].diff().dt.days
identities_df

,imo,source,shipname,flag,ssvid,match_fields,transmission_from,transmission_to,is_verified,days_since_last_update
170,9037123,self_reported,HAI CHANG DA LIAN,CHN,413202420,SEVERAL_FIELDS,2012-01-03 00:56:11+00:00,2015-05-26T10:00:35Z,True,NaN
169,9037123,self_reported,SHAN GANG RONG HE,CHN,413366530,SEVERAL_FIELDS,2015-05-26 10:21:35+00:00,2024-07-24T02:20:46Z,True,1239.0
168,9037123,registry,ZHONGGANGDALIAN,CHN,413366530,REGISTRY,2015-05-26 10:38:39+00:00,2019-12-14T08:37:28Z,True,0.0
173,9037123,self_reported,LIAOYINGYU 35669,NaN,415555555,NO_MATCH,2018-12-14 18:03:42+00:00,2022-09-21T08:08:42.25Z,False,1298.0
172,9037123,self_reported,ZHONG GANG DA LIAN,MHL,538090558,NO_MATCH,2019-12-14 09:12:33+00:00,2021-09-29T22:51:17Z,False,364.0
...,...,...,...,...,...,...,...,...,...,...
234,9867621,self_reported,ARISTOFANIS,MHL,538008714,NO_MATCH,2019-12-10 07:14:52+00:00,2024-09-17T03:01:32Z,False,NaN
235,9867621,registry,KHANKENDI,AZE,423543100,REGISTRY,2024-08-23 07:44:30+00:00,2026-07-31T13:30:53Z,True,1718.0
236,9867621,self_reported,KHANKENDI,AZE,423543100,SEVERAL_FIELDS,2024-08-23 07:44:37+00:00,2026-09-08T23:19:51Z,True,0.0
243,9884150,self_reported,SERGEY LVOV,RUS,273295620,NO_MATCH,2021-08-31 10:22:36+00:00,2026-09-08T23:26:26Z,False,NaN


In [9]:
before_2023 = identities_df[identities_df['transmission_from'] < '2023-01-01']
after_2023 = identities_df[identities_df['transmission_from'] >= '2023-01-01']

### CMP Table showing the average number of days before 2023 and in 2023

In [20]:
mean_before = before_2023.groupby('imo')['days_since_last_update'].mean()
mean_after = after_2023.groupby('imo')['days_since_last_update'].mean()
cmp_df = pd.DataFrame({'before': mean_before, 'after': mean_after}).dropna()
cmp_df

,before,after
imo,,
9037123,725.250000,800.333333
9164718,885.000000,1448.000000
9213296,448.500000,793.000000
9224453,1333.000000,652.500000
9233777,1083.666667,601.666667
9236248,904.750000,224.571429
9274525,799.800000,1283.000000
9282481,844.000000,453.166667
9290373,1254.000000,394.250000


In [12]:
faster = cmp_df[cmp_df['after'] < cmp_df['before']]
print(f"Started changing them more often: {len(faster)} из {len(cmp_df)} ships")

Started changing them more often: 15 из 37 ships


In [13]:
#save dataFrame to cache for visualization
cmp_df.reset_index().to_csv('../data/flag_change_intervals.csv', index=False)
identities_df.to_csv('../data/identities.csv', index=False)

In [16]:
identities_df['source'].value_counts()

source
self_reported    192
registry          57
Name: count, dtype: int64

In [33]:
mean_before = cmp_df['before'].mean()
mean_after = cmp_df['after'].mean()

print(f"Average before 2023: {mean_before:.1f} days")
print(f"Average after  2023: {mean_after:.1f} days")

Average before 2023: 828.6 days
Average after  2023: 1666.2 days


In [60]:
identities_df.groupby('imo')['flag'].nunique()
flags_before = before_2023.groupby('flag')['imo'].nunique()
flags_after = after_2023.groupby('flag')['imo'].nunique()

flags_df = pd.DataFrame({'before':flags_before, 'after':flags_after}).fillna(0).astype(int)
flags_df = flags_df.sort_values(by='after', ascending=False)
flags_df.to_csv('../data/flag_df.csv', index=False)

flags_df

,before,after
flag,,
PAN,4,23
CMR,0,10
LBR,9,9
RUS,12,9
SLE,1,6
BRB,0,5
MDG,0,4
GIN,0,4
MHL,19,3
